# Intro to TNR: Levin-Nave TRG

In [112]:
using TensorKit
import MatrixAlgebraKit as MAK
import TNRKit

In [135]:
function classical_ising(β::Real)
    c = sqrt(cosh(β))
    s = sqrt(sinh(β))
    M = TensorMap(Float64[c s; c -s], ℂ^2, ℂ^2)

    δ = zeros(Float64, ℂ^2 ⊗ ℂ^2, ℂ^2 ⊗ ℂ^2)
    for i in 1:2
        δ[i, i, i, i] = 1.0
    end

    @plansor T[-1 -2; -3 -4] := δ[1 2; 3 4] * adjoint(M)[-1; 1] * adjoint(M)[-2; 2] * M[3; -3] * M[4; -4]
    return T
end


mutable struct TRG{TT <: AbstractTensorMap}
    T::TT

    function TRG(T::TT) where {TT <: AbstractTensorMap}
        return new{TT}(T)
    end
end

In [139]:
function SVD12(scheme::TRG, trunc::MAK.TruncationStrategy)
    U, S, Vᴴ, _ = svd_trunc(scheme.T; trunc = trunc)
    return U * sqrt(S), sqrt(S) * Vᴴ
end


function step!(scheme::TRG, trunc::MAK.TruncationStrategy)
    A, B = SVD12(scheme.T, trunc)
    Tp = transpose(scheme.T, ((2, 4), (1, 3)))
    C, D = SVD12(Tp, trunc)
    @plansor scheme.T[-1 -2; -3 -4] := D[-2; 1 2] * B[-1; 4 1] * C[4 3; -3] * A[3 2; -4]
end


function finalize!(scheme::TRG)
    n = norm(@tensor scheme.T[1 2; 2 1])
    scheme.T /= n
    return n
end


function run!(scheme::TRG, trunc::MAK.TruncationStrategy, maxiter::Int)
    data = Vector{Float64}()
    push!(data, finalize!(scheme))

    steps = 0
    while steps < maxiter
        step!(scheme, trunc)
        push!(data, finalize!(scheme))
        steps += 1
    end
    return data
end

run! (generic function with 2 methods)

In [137]:
scheme = TRG(classical_ising(1.0))
data = run!(scheme, truncrank(8), 5)

6-element Vector{Float64}:
 14.7781121978613
  0.5001677313139515
  0.500670475341513
  0.49938298203670745
  0.499999860629703
  0.4999989533229532

In [138]:
scheme = TNRKit.TRG(TNRKit.classical_ising(Trivial, 1.0))
data = TNRKit.run!(scheme, truncrank(8), TNRKit.maxiter(5))

┌ Info: Starting simulation
│  TRG - Tensor Renormalization Group
│   * T: 2×2←2×2 TensorMap{Float64, ComplexSpace, 2, 2, Vector{Float64}}
└ 
┌ Info: Simulation finished
│  Maximum amount of iterations reached: 5
│  Elapsed time: 0.005544188s
└  Iterations: 5


6-element Vector{Float64}:
 14.7781121978613
  0.5001677313139515
  0.500670475341513
  0.4993829820367073
  0.49999986062970364
  0.49999895332295263